# rPPG — 7. tur: aynı veriyle A/B karşılaştırması (iki T4, paralel)

Kaggle'ın iki GPU'su aynı veri üzerinde iki eğitimi **aynı anda** çalıştırır; tek fark eğitim ayarlarıdır:

* **A (kontrol, GPU 0):** 6. turun ayarları aynen.
* **B (yeni, GPU 1):** A + `--phone-aug --spec-loss 0.2 --ema 0.999 --val-viterbi --amp --speed 0.7 1.5 --hr-balance 3`
  (telefon bozulmaları, frekans alanı kaybı, ağırlık EMA'sı, Viterbi ile seçim, karışık hassasiyet, geniş hız artırması, nabız dengeli örnekleme).

İki model de en son turun modelinden (varsa 6. tur, yoksa 5. tur, son epoch) başlar. Sonda ana testte ve telefon
testlerinde yan yana ölçülür; B ayrıca örtüşmeli çıkarım ve yatay çevirmeyle (`--tta`) de ölçülür.
B'deki seçenekler paket olarak denenir; B kazanırsa sonraki turda tek tek kapatılarak hangisinin işe yaradığı ayrılabilir.
`--amp` B'yi hızlandırır, bu yüzden aynı sürede B daha çok epoch görebilir (günlükte epoch sayıları yazılır).

**Kod dalı:** yeni seçenekler `main`'e birleşene kadar 1. hücredeki `BRANCH` `claude/iyilestirmeler` kalmalı.

**Gerekenler:**
0. **Add Input** → arama: `rppg-dlcn` → **dalaoplan/rppg-dlcn** veri setini ekleyin (Hugging Face onayı gerekmez).
1. Hugging Face hesabıyla şu üç sayfada erişim isteği gönderilmiş ve **onaylanmış** olmalı:
   [mini_MMPD](https://huggingface.co/datasets/WeiQian98/mini_MMPD),
   [VIPL-HR](https://huggingface.co/datasets/WeiQian98/VIPL-HR),
   [COHFACE](https://huggingface.co/datasets/WeiQian98/COHFACE).
2. Kaggle **Add-ons → Secrets** altında `HF_TOKEN` bu not defterine bağlı olmalı.
3. **Settings:** Accelerator **GPU T4 x2**, **Turn on internet**.

DLCN eklenmemişse ya da erişimi onaylanmamış veri seti atlanır, eğitim diğer verilerle yine yapılır. 2. hücre her veri setinden bir videoyu
hemen işleyip okumanın çalıştığını gösterir; orada hata görürseniz çalıştırmayı durdurup haber verin.

**Çalıştırma:** **Save Version → Save & Run All (Commit)**; ≈11 saat. Bitince **Output** → `cikti` klasörünü indirin.

In [ ]:
# 1) Ortam, kod ve rPPG-Toolbox
import os, subprocess, shutil, sys, time, glob, json
from datetime import datetime, timedelta

T0 = datetime.now()
print(subprocess.run(["nvidia-smi"], capture_output=True, text=True).stdout)
assert shutil.which("nvidia-smi"), "GPU yok: Settings -> Accelerator: GPU T4 x2"
assert subprocess.run(["curl", "-sI", "https://github.com"], capture_output=True).returncode == 0, \
    "İnternet kapalı: Settings -> Turn on internet"

REPO = "/kaggle/working/rppg"
BRANCH = "claude/iyilestirmeler"   # yeni eğitim seçenekleri bu dalda; main'e birleşince "main"
if not os.path.exists(REPO):
    subprocess.run(["git", "clone", "--depth", "1", "--branch", BRANCH, "https://github.com/Nisagwn/rppg.git", REPO], check=True)
TB = os.path.join(REPO, "external", "rPPG-Toolbox")
if not os.path.exists(TB):
    subprocess.run(["git", "clone", "--depth", "1", "https://github.com/ubicomplab/rPPG-Toolbox.git", TB], check=True)
os.chdir(REPO)

try:
    from kaggle_secrets import UserSecretsClient
    os.environ["HF_TOKEN"] = UserSecretsClient().get_secret("HF_TOKEN")
    print("HF_TOKEN var")
except Exception:
    print("!" * 70)
    print("UYARI: HF_TOKEN yok, MMPD / VIPL-HR / COHFACE atlanacak. Add-ons -> Secrets -> HF_TOKEN kutusunu "
          "işaretleyin ve not defterini yeniden başlatın.")
    print("!" * 70)

import torch
print("torch", torch.__version__, "| cuda", torch.cuda.is_available(), "| GPU sayısı", torch.cuda.device_count())
N_GPU = torch.cuda.device_count()
if N_GPU < 2:
    print("UYARI: tek GPU; A ve B aynı GPU'yu paylaşacak (yavaş). Settings -> Accelerator: GPU T4 x2")

In [ ]:
# 2) Erişim ve okuma denemesi: her yeni veri setinden bir video (birkaç dakika)
PY = sys.executable
env = dict(os.environ, PYTHONUNBUFFERED="1")
r = subprocess.run([PY, "-u", "scripts/dl_prepare.py", "--out", "/tmp/dl_deneme", "--raw", "/tmp/dl_deneme_raw",
                    "--dlcn", "1", "--cohface", "1", "--mmpd", "1", "--vipl", "1", "--workers", "1"],
                   cwd=REPO, capture_output=True, text=True, env=env)
print(r.stdout[-4000:], r.stderr[-3000:])
print("işlenen:", sorted(os.path.basename(p) for p in glob.glob("/tmp/dl_deneme/*/*.npz")))

In [ ]:
# 3) Klasörler, süre, başlangıç modeli
DATA, RAW = "/tmp/dl", "/tmp/dl_raw"
OUT = "/kaggle/working/cikti"
OUT_A, OUT_B = os.path.join(OUT, "A_kontrol"), os.path.join(OUT, "B_yeni")
LOGS = os.path.join(OUT, "gunlukler")
for d in (DATA, RAW, OUT, OUT_A, OUT_B, LOGS):
    os.makedirs(d, exist_ok=True)

TRAIN_UNTIL = T0 + timedelta(hours=12 - 2.3)          # Kaggle oturum sınırı 12 saat; iki modelin değerlendirme payı
CANDIDATES = [("6. tur", "results/derin_ogrenme/kaggle_tur6/last.pth"),
              ("5. tur", "results/derin_ogrenme/kaggle_tur5/last.pth")]
INIT_NAME, INIT = next((n, os.path.join(REPO, p)) for n, p in CANDIDATES if os.path.exists(os.path.join(REPO, p)))
TUR3 = os.path.join(REPO, "results/derin_ogrenme/kaggle_tur3/last.pth")   # uygulamadaki model
print(f"başlangıç: {INIT_NAME} ({INIT})")
print(f"/tmp boş {shutil.disk_usage('/tmp').free / 1e9:.0f} GB | eğitim en geç {TRAIN_UNTIL:%H:%M}")


In [ ]:
# 4) Veri hazırlama (arka planda, indir-işle-sil)
#    A: MCD-rPPG 600 kişi   B: test kümesi -> UBFC 34 -> PURE 60   C: DLCN (Kaggle girdisi)
#    D: MMPD, VIPL-HR (önce telefon kaynağı), COHFACE — paralel (Hugging Face; test kişileri önce)
def launch(name, cmd):
    return subprocess.Popen(["bash", "-c", cmd], cwd=REPO, env=env,
                            stdout=open(os.path.join(LOGS, name + ".log"), "a"), stderr=subprocess.STDOUT)

P = f"{PY} -u scripts/dl_prepare.py --out {DATA} --raw {RAW}"
jobs = {
    "hazirla_mcd": launch("hazirla_mcd", f"{P} --mcd 600 --workers 2 --max-sec 45"),
    "hazirla_test_ubfc_pure": launch("hazirla_test_ubfc_pure",
        f"{P} --test-download --workers 2; {P} --ubfc 34 --workers 1; {P} --pure 60 --workers 1"),
    "hazirla_dlcn": launch("hazirla_dlcn", f"{P} --dlcn 784 --workers 1"),
    "hazirla_mmpd": launch("hazirla_mmpd", f"{P} --mmpd 660 --workers 1"),
    "hazirla_vipl": launch("hazirla_vipl", f"{P} --vipl 2500 --workers 1 --max-sec 60"),
    "hazirla_cohface": launch("hazirla_cohface", f"{P} --cohface 164 --workers 1"),
}
print("veri hazırlama başladı")

In [ ]:
# 5) Eğitim: 300 video hazır olunca A ve B başlar (ayrı GPU'lar); yeni gelen veri her epoch başında eklenir
SUBS = ("mcd", "ubfc", "pure", "dlcn", "mmpd", "vipl", "cohface")
TESTS = ("test", "test_mmpd", "test_vipl", "test_dlcn", "test_cohface")
def counts(subs=SUBS):
    return {s: len(glob.glob(os.path.join(DATA, s, "*.npz"))) for s in subs}

while sum(counts().values()) < 300 and any(j.poll() is None for j in jobs.values()):
    print(time.strftime("%H:%M"), counts(), flush=True)
    time.sleep(120)

BASE = [PY, "-u", "scripts/dl_train.py", "--data", DATA, "--init", INIT, "--epochs", "100", "--steps", "500",
        "--lr", "1e-4", "--deadline", TRAIN_UNTIL.strftime("%Y-%m-%d %H:%M")]
NEW = ["--phone-aug", "--spec-loss", "0.2", "--ema", "0.999", "--val-viterbi", "--amp", "--speed", "0.7", "1.5", "--hr-balance", "3"]

def train_job(name, out, extra, gpu):
    e = dict(env, CUDA_VISIBLE_DEVICES=str(gpu % max(N_GPU, 1)))
    return subprocess.Popen(BASE + ["--out", out] + extra, cwd=REPO, env=e,
                            stdout=open(os.path.join(LOGS, name + ".log"), "a"), stderr=subprocess.STDOUT)

trains = {"egitim_A": train_job("egitim_A", OUT_A, [], 0), "egitim_B": train_job("egitim_B", OUT_B, NEW, 1)}
print("eğitim başladı: A (kontrol) GPU 0, B (yeni) GPU 1")


In [ ]:
# 6) İzleme: 5 dakikada bir özet (A ve B'nin son epoch satırı)
def tail(path, n):
    try:
        return open(path, encoding="utf-8", errors="replace").read().splitlines()[-n:]
    except OSError:
        return []

def free_ram():
    try:
        import psutil
        return f"RAM boş {psutil.virtual_memory().available / 1e9:.1f} GB"
    except Exception:
        return ""

def last_epoch(name):
    ep = [l for l in tail(os.path.join(LOGS, name + ".log"), 400) if l.startswith("epoch ")]
    return ep[-1] if ep else "ilk epoch sürüyor"

while any(t.poll() is None for t in trains.values()):
    try:
        print(f"{time.strftime('%H:%M')} | {counts()} | test {counts(TESTS)} | disk {shutil.disk_usage('/tmp').free/1e9:.0f} GB"
              f" | {free_ram()}\n   A: {last_epoch('egitim_A')}\n   B: {last_epoch('egitim_B')}", flush=True)
    except Exception as e:
        print("izleme hatası:", repr(e), flush=True)
    time.sleep(300)
for name in trains:
    print(f"===== {name}\n" + "\n".join(tail(os.path.join(LOGS, name + ".log"), 12)))


In [ ]:
# 7) Veri kalitesi özeti (eğitimde kullanılan filtre; kalite.csv çıktıya kopyalanır)
r = subprocess.run([PY, "scripts/dl_quality.py", "--data", DATA, "--workers", "4", "--sources", *SUBS],
                   cwd=REPO, capture_output=True, text=True, env=env)
print(r.stdout[-3000:], r.stderr[-2000:])
if os.path.exists(os.path.join(DATA, "kalite.csv")):
    shutil.copy(os.path.join(DATA, "kalite.csv"), OUT)

In [ ]:
# 8) Değerlendirme: B (yeni) ana model; A (kontrol), başlangıç ve uygulamadaki model aynı videolarda karşılaştırılır.
#    B ayrıca örtüşmeli çıkarım + yatay çevirmeyle (--tta) ölçülür.
for j in jobs.values():
    if j.poll() is None:
        j.terminate()
print("test videoları:", counts(TESTS))
for t in TESTS:
    d = os.path.join(DATA, t)
    if not glob.glob(os.path.join(d, "*.npz")):
        continue
    out = OUT_B if t == "test" else os.path.join(OUT_B, t)
    r = subprocess.run([PY, "scripts/dl_evaluate.py", "--model", os.path.join(OUT_B, "best.pth"), "--test", d,
                        "--data", DATA, "--out", out, "--history", os.path.join(OUT_B, "gecmis.json"), "--tta",
                        "--compare", f"A kontrol (en iyi)={os.path.join(OUT_A, 'best.pth')}",
                        f"A kontrol (son epoch)={os.path.join(OUT_A, 'last.pth')}",
                        f"{INIT_NAME} (başlangıç)={INIT}", f"3. tur (uygulamadaki)={TUR3}"],
                       cwd=REPO, capture_output=True, text=True, env=env)
    print(f"===== {t}\n", r.stdout[-5000:], r.stderr[-2000:])
for name, o in (("A", OUT_A), ("B", OUT_B)):
    if os.path.exists(os.path.join(o, "gecmis.json")):
        h = json.load(open(os.path.join(o, "gecmis.json")))
        print(f"{name}: {h['epoch']} epoch, en iyi doğrulama {h['best']['val_mae']:.2f} BPM (epoch {h['best']['epoch']})")


In [ ]:
# 9) Çıktılar
for p in sorted(glob.glob(os.path.join(OUT, "**"), recursive=True)):
    if os.path.isfile(p):
        print(f"{os.path.getsize(p)/1e6:8.2f} MB  {p}")
shutil.rmtree(REPO, ignore_errors=True)   # çıktıda yalnızca cikti/ kalsın
print("süre:", datetime.now() - T0)